# LeetCode 573 — Squirrel Simulation bằng Python

Link đề: https://leetcode.com/problems/squirrel-simulation/ · Độ khó: **Medium** · Chủ đề: Array, Math (khoảng cách Manhattan, tối ưu hóa bằng công thức)

> **Bài Premium.** Đề trong notebook này được viết theo hiểu biết về bài (có đối chiếu với lời giải tham khảo công khai). Nếu có tài khoản Premium, hãy đối chiếu lại đề gốc trên LeetCode, đặc biệt là phần ràng buộc về toạ độ.

Skeleton LeetCode:

```python
class Solution:
    def minDistance(self, height: int, width: int, tree: List[int], squirrel: List[int], nuts: List[List[int]]) -> int:
        ...
```

Ghi chú: bài nhìn như một bài "mô phỏng đường đi" hay "thứ tự nhặt hạt" (có vẻ rất khó, kiểu TSP), nhưng thật ra chỉ cần **một công thức** và **một vòng lặp**. Notebook này đi từ cách vét cạn mọi thứ tự nhặt, qua cách thử từng hạt đầu tiên, tới công thức `O(n)`.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: một chú sóc phải đem **tất cả** các hạt về gốc cây, **mỗi lần chỉ mang một hạt**, di chuyển 4 hướng trên lưới; cần tìm tổng số bước **ít nhất**.
2. Nắm các công cụ Python cần dùng: `abs`, khoảng cách Manhattan, gỡ tuple `for r, c in nuts`, generator expression với `sum`/`min`, `itertools.permutations`, và `float("inf")`.
3. Đi từ **vét cạn mọi thứ tự nhặt** → **thử từng hạt làm hạt đầu và mô phỏng** → **công thức hai lượt** → **một lượt duyệt**, và hiểu vì sao các cách cho cùng đáp án.
4. Hiểu mấu chốt: **chỉ có hạt đầu tiên là "đặc biệt"**; mọi hạt còn lại luôn tốn đúng `2 × (khoảng cách hạt → cây)`. Vì vậy bài toán chỉ còn là chọn hạt đầu sao cho "tiết kiệm được nhiều nhất".
5. Tránh các lỗi kinh điển: quên nhân đôi, chọn hạt gần sóc nhất một cách tham lam (sai!), chỉ so khoảng cách sóc → hạt mà quên trừ đoạn hạt → cây, và quên rằng sóc kết thúc ở **cây**.

## 2. Đề bài & ví dụ

Có một khu vườn kích thước `height x width`. Cho:

- `tree = [tree_r, tree_c]`: vị trí cây,
- `squirrel = [squirrel_r, squirrel_c]`: vị trí ban đầu của sóc,
- `nuts`: danh sách các hạt, `nuts[i] = [r, c]`.

Sóc di chuyển lên/xuống/trái/phải sang ô kề, **mỗi lần chỉ mang tối đa một hạt**. Hãy trả về **tổng số bước ít nhất** để sóc nhặt **mọi hạt** và đặt từng hạt dưới gốc cây (nhặt hạt, đem về cây, quay ra nhặt hạt khác, ...). Khoảng cách chính là số bước di chuyển.

**Ví dụ 1**

- Input: `height = 5, width = 7, tree = [2,2], squirrel = [4,4], nuts = [[3,0],[2,5]]`
- Output: `12`
- Giải thích: sóc nên đi nhặt hạt `[2,5]` trước thì tổng số bước là ít nhất.

**Ví dụ 2**

- Input: `height = 1, width = 3, tree = [0,1], squirrel = [0,0], nuts = [[0,2]]`
- Output: `3`

**Ràng buộc:**

- `1 <= height, width <= 100`
- `tree.length == 2`, `squirrel.length == 2`
- `1 <= nuts.length <= 5000`, `nuts[i].length == 2`
- `0 <= tree_r, squirrel_r, nuts[i][0] <= height`
- `0 <= tree_c, squirrel_c, nuts[i][1] <= width`

Lưu ý: ràng buộc toạ độ trong đề gốc ghi là `<= height` / `<= width` (chứ không phải `< height`), nên ta không dựa vào kích thước lưới để làm gì cả — và thật ra **`height`, `width` không cần dùng** trong lời giải. Đề không có chướng ngại vật, nên đường đi ngắn nhất giữa hai ô luôn là khoảng cách Manhattan. (Phần ràng buộc này: đối chiếu lại đề gốc nếu có tài khoản Premium.)

In [1]:
from typing import List, Tuple
from itertools import permutations
import random

# Hai ví dụ của đề, lưu thành danh sách để tái sử dụng
examples = [
    # (height, width, tree, squirrel, nuts, expected)
    (5, 7, [2, 2], [4, 4], [[3, 0], [2, 5]], 12),
    (1, 3, [0, 1], [0, 0], [[0, 2]], 3),
]
for h, w, tree, sq, nuts, expected in examples:
    print(f"lưới {h}x{w} | cây {tree} | sóc {sq} | {len(nuts)} hạt | kỳ vọng {expected}")

lưới 5x7 | cây [2, 2] | sóc [4, 4] | 2 hạt | kỳ vọng 12
lưới 1x3 | cây [0, 1] | sóc [0, 0] | 1 hạt | kỳ vọng 3


## 3. Kiến thức nền

### 3.1. Khoảng cách Manhattan và `abs`

Trên lưới, mỗi bước chỉ đi sang ô kề (lên/xuống/trái/phải). Để đi từ ô `(r1, c1)` tới ô `(r2, c2)` ta cần đi lệch `|r1 - r2|` hàng và `|c1 - c2|` cột. Không có đường chéo, nên số bước ít nhất là tổng của hai đại lượng đó. Đó là **khoảng cách Manhattan** (đặt theo các dãy phố vuông vức của Manhattan).

`abs(x)` là giá trị tuyệt đối: `abs(-3) == 3`, `abs(4) == 4`.

Ví dụ: từ `(4, 4)` tới `(2, 5)` cần `|4-2| + |4-5| = 2 + 1 = 3` bước. Vì không có chướng ngại vật, **luôn tồn tại** đường đi đúng bằng số đó (đi hết chiều dọc rồi hết chiều ngang), và không thể ít hơn.

In [2]:
def manhattan(p, q):
    # khoảng cách Manhattan giữa hai ô p = [r, c] và q = [r, c]
    return abs(p[0] - q[0]) + abs(p[1] - q[1])

print(manhattan([4, 4], [2, 5]))   # 3
print(manhattan([2, 2], [3, 0]))   # 1 + 2 = 3
print(manhattan([2, 2], [2, 2]))   # 0 (cùng một ô)
print(manhattan([0, 0], [0, 2]))   # 2

# Đối xứng: đi từ p tới q hay từ q tới p đều tốn số bước như nhau
assert manhattan([4, 4], [2, 5]) == manhattan([2, 5], [4, 4])

3
3
0
2


### 3.2. Gỡ tuple/list trong vòng `for`

`nuts` là danh sách các cặp `[r, c]`. Thay vì viết `nut[0]`, `nut[1]`, Python cho phép **gỡ (unpack)** ngay ở đầu vòng lặp:

```python
for r, c in nuts:
    ...
```

Tương tự `tr, tc = tree` gán `tr = tree[0]`, `tc = tree[1]`. Điều kiện: mỗi phần tử phải có đúng 2 giá trị (đề đảm bảo `length == 2`).

In [3]:
nuts = [[3, 0], [2, 5]]
tree = [2, 2]

tr, tc = tree
print("tr =", tr, ", tc =", tc)

for r, c in nuts:
    print("hạt tại hàng", r, "cột", c, "-> cách cây", abs(r - tr) + abs(c - tc))

tr = 2 , tc = 2
hạt tại hàng 3 cột 0 -> cách cây 3
hạt tại hàng 2 cột 5 -> cách cây 3


### 3.3. Generator expression với `sum` và `min`

Thay vì tạo danh sách trung gian, ta có thể đưa thẳng một biểu thức sinh (generator expression) vào `sum` hoặc `min`:

```python
sum(abs(r - tr) + abs(c - tc) for r, c in nuts)
min(f(x) for x in danh_sach)
```

- `sum(...)` cộng tất cả giá trị sinh ra.
- `min(...)` lấy giá trị nhỏ nhất. Nếu dãy **rỗng** thì `min` báo lỗi `ValueError` (đề đảm bảo có ít nhất 1 hạt nên không lo, nhưng cần nhớ khi tự viết).

In [4]:
nuts = [[3, 0], [2, 5]]
tr, tc = 2, 2

dist_to_tree = [abs(r - tr) + abs(c - tc) for r, c in nuts]   # list comprehension: để xem từng giá trị
print("khoảng cách từng hạt tới cây:", dist_to_tree)

total = sum(abs(r - tr) + abs(c - tc) for r, c in nuts)       # generator: không tạo list
print("tổng:", total)

print("nhỏ nhất:", min(abs(r - tr) + abs(c - tc) for r, c in nuts))

try:
    min(x for x in [])
except ValueError as e:
    print("min của dãy rỗng ->", type(e).__name__)

khoảng cách từng hạt tới cây: [3, 3]
tổng: 6
nhỏ nhất: 3
min của dãy rỗng -> ValueError


### 3.4. `itertools.permutations` — mọi thứ tự nhặt hạt

`permutations([0, 1, 2])` sinh ra mọi cách sắp xếp thứ tự của 3 phần tử: có `3! = 6` hoán vị. Ta sẽ dùng nó để **vét cạn mọi thứ tự nhặt hạt** làm "chuẩn vàng" (oracle) kiểm chứng công thức ở các input nhỏ. Với `n` hạt có `n!` thứ tự, nên chỉ dùng được khi `n` nhỏ (ví dụ `n <= 7`).

In [5]:
from itertools import permutations
from math import factorial

print(list(permutations([0, 1, 2])))
print(len(list(permutations(range(5)))), "==", factorial(5))

[(0, 1, 2), (0, 2, 1), (1, 0, 2), (1, 2, 0), (2, 0, 1), (2, 1, 0)]
120 == 120


### 3.5. `float("inf")` làm giá trị khởi tạo cho `min`

Khi tìm giá trị nhỏ nhất bằng cách cập nhật dần, ta khởi tạo `best = float("inf")` (vô cực dương: lớn hơn mọi số). Lần so sánh đầu tiên `min(best, x)` sẽ luôn cho `x`.

Lưu ý: `float("inf")` có kiểu `float`. Nếu mọi cập nhật đều là `int` thì kết quả cuối vẫn là `int` vì `min` trả về chính đối tượng nhỏ hơn; còn khi `nuts` rỗng ta sẽ nhận lại `inf` (kiểu float) — đề không cho phép trường hợp đó.

In [6]:
best = float("inf")
for x in [7, 3, 9]:
    best = min(best, x)
print(best, type(best).__name__)   # 3 int

print(float("inf") > 10**18)       # True

3 int
True


## 4. Ý tưởng nền tảng / trực giác

Hãy tính tay Ví dụ 1: cây `T = (2,2)`, sóc `S = (4,4)`, hạt `A = (3,0)`, `B = (2,5)`.

Mọi con đường của sóc đều có dạng:

`S → (hạt đầu) → T → (hạt thứ hai) → T → ... → T`.

Tính khoảng cách cần thiết:

- `d(T, A) = |2-3| + |2-0| = 3`; `d(T, B) = |2-2| + |2-5| = 3`.
- `d(S, A) = |4-3| + |4-0| = 5`; `d(S, B) = |4-2| + |4-5| = 3`.

**Nhặt A trước, rồi B:** `S→A` (5) + `A→T` (3) + `T→B` (3) + `B→T` (3) = **14**.

**Nhặt B trước, rồi A:** `S→B` (3) + `B→T` (3) + `T→A` (3) + `A→T` (3) = **12**.

Hai kết quả khác nhau chỉ ở **hạt đầu tiên**. Hạt thứ hai luôn tốn một "chuyến khứ hồi" cây → hạt → cây, tức `2 × d(T, hạt)`, bất kể hạt đó ở đâu.

In [7]:
T, S = [2, 2], [4, 4]
A, B = [3, 0], [2, 5]

d = manhattan
a_first = d(S, A) + d(A, T) + d(T, B) + d(B, T)
b_first = d(S, B) + d(B, T) + d(T, A) + d(A, T)
print("nhặt A trước:", a_first)   # 14
print("nhặt B trước:", b_first)   # 12
print("đáp án =", min(a_first, b_first))

nhặt A trước: 14
nhặt B trước: 12
đáp án = 12


## 5. Cách 1 — Brute Force: thử mọi thứ tự nhặt

### Ý tưởng

Đề hỏi "nhặt theo thứ tự nào ngắn nhất", vậy cứ thử **tất cả** các thứ tự. Với mỗi hoán vị của các hạt, mô phỏng đường đi:

1. Sóc ở `squirrel`, đi tới hạt đầu tiên của hoán vị, rồi về cây.
2. Với mỗi hạt tiếp theo: từ cây tới hạt, từ hạt về cây.
3. Cộng dồn số bước; lấy tổng nhỏ nhất qua mọi hoán vị.

### Code

In [8]:
def min_distance_brute(height, width, tree, squirrel, nuts):
    best = float("inf")
    for order in permutations(range(len(nuts))):
        pos = squirrel          # sóc đang đứng ở đâu
        steps = 0
        for idx in order:
            nut = nuts[idx]
            steps += manhattan(pos, nut)      # đi tới hạt
            steps += manhattan(nut, tree)     # đem hạt về cây
            pos = tree                        # sau mỗi chuyến, sóc đứng ở cây
        best = min(best, steps)
    return best

for h, w, tree, sq, nuts, expected in examples:
    got = min_distance_brute(h, w, tree, sq, nuts)
    print(got, expected, "OK" if got == expected else "SAI")

12 12 OK
3 3 OK


### Độ phức tạp và nút thắt

- **Time:** `O(n! · n)` — có `n!` hoán vị, mỗi hoán vị mô phỏng `n` hạt. Với `n = 5000` hoàn toàn bất khả thi (ngay `n = 15` đã là hàng nghìn tỷ hoán vị).
- **Space:** `O(n)` cho một hoán vị đang xét (generator `permutations` không lưu tất cả).

**Nút thắt:** ta đang phân biệt các thứ tự khác nhau, trong khi quan sát ở mục 4 cho thấy phần lớn thứ tự cho ra con số *giống hệt nhau*. Chỉ hạt đầu tiên làm thay đổi tổng. Cách này vẫn rất hữu ích: nó là **oracle** (đáp án chuẩn độc lập) để kiểm tra các cách nhanh hơn ở input nhỏ.

## 6. Ý tưởng thuật toán / bước đột phá

Gọi, với mỗi hạt `i`:

- `a_i = d(cây, hạt i)` — khoảng cách hạt tới cây,
- `b_i = d(sóc, hạt i)` — khoảng cách sóc ban đầu tới hạt.

**Quan sát then chốt:** sau chuyến đầu tiên, sóc luôn đứng ở cây. Vì vậy:

- Hạt **không phải đầu tiên**: tốn `cây → hạt → cây = 2·a_i`.
- Hạt **đầu tiên** (giả sử là hạt `k`): tốn `sóc → hạt k → cây = b_k + a_k`.

Gọi `S = 2 · Σ a_i` (tổng nếu **mọi** hạt đều tốn `2a`). Nếu hạt `k` được nhặt đầu tiên thì tổng thực tế là:

```
tổng(k) = S − 2·a_k + (b_k + a_k) = S − a_k + b_k
```

Đáp án = `min` qua mọi `k` của `S − a_k + b_k`, tức là:

```
đáp án = S + min_k (b_k − a_k)  =  S − max_k (a_k − b_k)
```

Nói cách khác: nếu sóc chọn hạt `k` làm hạt đầu, nó **tiết kiệm** được `a_k − b_k` bước so với trường hợp lý tưởng "sóc cũng bắt đầu từ cây". Muốn tổng ngắn nhất thì chọn hạt **tiết kiệm nhiều nhất** (có thể âm: khi đó mọi hạt đều làm xấu đi, ta chọn hạt ít xấu nhất).

In [9]:
# Ví dụ 1: tính a_i, b_i, S và tổng(k) cho từng hạt
tree, sq = [2, 2], [4, 4]
nuts = [[3, 0], [2, 5]]

a = [manhattan(tree, n) for n in nuts]
b = [manhattan(sq, n) for n in nuts]
S_total = 2 * sum(a)
print("a =", a, "| b =", b, "| S =", S_total)
for k in range(len(nuts)):
    print(f"hạt đầu = {nuts[k]}: tổng = S - a + b = {S_total} - {a[k]} + {b[k]} = {S_total - a[k] + b[k]}")

a = [3, 3] | b = [5, 3] | S = 12
hạt đầu = [3, 0]: tổng = S - a + b = 12 - 3 + 5 = 14
hạt đầu = [2, 5]: tổng = S - a + b = 12 - 3 + 3 = 12


## 7. Các cách tiếp theo

### 7.1. Cách 2 — Thử từng hạt làm hạt đầu và mô phỏng đường đi (`O(n²)`)

**Giải thích từng đoạn.**

1. Thay vì thử `n!` thứ tự, ta chỉ thử `n` lựa chọn: hạt nào là hạt đầu tiên.
2. Với mỗi lựa chọn, thứ tự các hạt còn lại không ảnh hưởng (mỗi hạt tốn `2a`). Ta mô phỏng bằng cách cộng `b_k + a_k` rồi cộng `2a_i` cho mọi hạt `i ≠ k`.
3. Lấy `min` qua `k`.

Cách này **chưa dùng** công thức `S − a_k + b_k`: nó cộng lại từng hạt khác, nên tốn `O(n)` mỗi lần chọn → `O(n²)`. Với `n = 5000` là 25 triệu phép tính: chạy được nhưng chậm trong Python.

**Code demo:**

In [10]:
def min_distance_try_first(height, width, tree, squirrel, nuts):
    n = len(nuts)
    best = float("inf")
    for k in range(n):                       # thử hạt k làm hạt đầu
        total = manhattan(squirrel, nuts[k]) + manhattan(nuts[k], tree)
        for i in range(n):
            if i != k:
                total += 2 * manhattan(nuts[i], tree)   # các hạt còn lại: khứ hồi cây-hạt
        best = min(best, total)
    return best

for h, w, tree, sq, nuts, expected in examples:
    print(min_distance_try_first(h, w, tree, sq, nuts), expected)

12 12
3 3


**Code nộp LeetCode (Cách 2):**

In [11]:
class SolutionTryFirst:
    def minDistance(self, height: int, width: int, tree: List[int], squirrel: List[int], nuts: List[List[int]]) -> int:
        tr, tc = tree
        sr, sc = squirrel
        n = len(nuts)
        best = float("inf")
        for k in range(n):
            r, c = nuts[k]
            total = abs(r - sr) + abs(c - sc) + abs(r - tr) + abs(c - tc)
            for i in range(n):
                if i != k:
                    r2, c2 = nuts[i]
                    total += 2 * (abs(r2 - tr) + abs(c2 - tc))
            best = min(best, total)
        return best

print(SolutionTryFirst().minDistance(5, 7, [2, 2], [4, 4], [[3, 0], [2, 5]]))   # 12

12


**Độ phức tạp (Cách 2):** Time `O(n²)`, Space `O(1)`. Nút thắt: ta cộng đi cộng lại cùng một tổng `Σ 2a_i` cho mỗi `k`. Chỉ cần tính tổng đó **một lần**.

### 7.2. Cách 3 — Công thức hai lượt `O(n)` (lời giải chuẩn)

**Giải thích từng đoạn.**

1. Lượt 1: tính `S = 2 · Σ a_i` bằng một vòng `sum`.
2. Lượt 2: với mỗi hạt, tính `S − a + b`, giữ giá trị nhỏ nhất.
3. Trả về giá trị đó.

Chỉ hai vòng lặp qua `nuts`, không lồng nhau.

**Code demo:**

In [12]:
def min_distance_formula(height, width, tree, squirrel, nuts):
    tr, tc = tree
    sr, sc = squirrel
    s = sum(abs(r - tr) + abs(c - tc) for r, c in nuts) * 2     # lượt 1
    ans = float("inf")
    for r, c in nuts:                                           # lượt 2
        a = abs(r - tr) + abs(c - tc)       # hạt -> cây
        b = abs(r - sr) + abs(c - sc)       # sóc -> hạt
        ans = min(ans, s - a + b)
    return ans

for h, w, tree, sq, nuts, expected in examples:
    print(min_distance_formula(h, w, tree, sq, nuts), expected)

12 12
3 3


**Code nộp LeetCode (Cách 3):**

In [13]:
class SolutionFormula:
    def minDistance(self, height: int, width: int, tree: List[int], squirrel: List[int], nuts: List[List[int]]) -> int:
        tr, tc = tree
        sr, sc = squirrel
        s = sum(abs(r - tr) + abs(c - tc) for r, c in nuts) * 2
        ans = float("inf")
        for r, c in nuts:
            a = abs(r - tr) + abs(c - tc)
            b = abs(r - sr) + abs(c - sc)
            ans = min(ans, s - a + b)
        return ans

print(SolutionFormula().minDistance(1, 3, [0, 1], [0, 0], [[0, 2]]))   # 3

3


**Độ phức tạp (Cách 3):** Time `O(n)`, Space `O(1)`.

### 7.3. Cách 4 — Một lượt duyệt: giữ "mức tiết kiệm lớn nhất"

**Giải thích từng đoạn.** Ta viết lại công thức dưới dạng `đáp án = S − max_k (a_k − b_k)`. Khi đó trong **một** vòng `for`, ta vừa cộng dồn `2a` vào `total`, vừa cập nhật `saving = max(saving, a − b)`. Hết vòng lặp, trả về `total − saving`. Cách này không cần biết `S` trước nên không cần hai lượt (hữu ích nếu dữ liệu đến dạng luồng, đọc một lần).

Chú ý: `saving` phải khởi tạo là **vô cực âm** `-inf` chứ không phải `0`, vì `a − b` có thể âm với mọi hạt (ví dụ hạt nằm sát cây còn sóc đứng rất xa: `a − b < 0`).

**Code demo:**

In [14]:
def min_distance_one_pass(height, width, tree, squirrel, nuts):
    tr, tc = tree
    sr, sc = squirrel
    total = 0
    saving = float("-inf")
    for r, c in nuts:
        a = abs(r - tr) + abs(c - tc)
        b = abs(r - sr) + abs(c - sc)
        total += 2 * a
        saving = max(saving, a - b)
    return total - saving

for h, w, tree, sq, nuts, expected in examples:
    print(min_distance_one_pass(h, w, tree, sq, nuts), expected)

# Trường hợp 'saving' âm: hạt sát cây, sóc ở xa
print(min_distance_one_pass(10, 10, [0, 0], [9, 9], [[0, 1]]))   # b=17, a=1 -> 17 + 1 = 18

12 12
3 3
18


**Code nộp LeetCode (Cách 4):**

In [15]:
class SolutionOnePass:
    def minDistance(self, height: int, width: int, tree: List[int], squirrel: List[int], nuts: List[List[int]]) -> int:
        tr, tc = tree
        sr, sc = squirrel
        total = 0
        saving = float("-inf")
        for r, c in nuts:
            a = abs(r - tr) + abs(c - tc)
            b = abs(r - sr) + abs(c - sc)
            total += 2 * a
            saving = max(saving, a - b)
        return total - saving

print(SolutionOnePass().minDistance(5, 7, [2, 2], [4, 4], [[3, 0], [2, 5]]))   # 12

12


**Độ phức tạp (Cách 4):** Time `O(n)`, Space `O(1)`.

### 7.4. Cách 5 — Sắp xếp theo mức tiết kiệm (chỉ để thấy vì sao không cần)

**Giải thích.** Nếu tính `saving_i = a_i − b_i` cho mọi hạt rồi **sắp xếp giảm dần**, hạt đầu danh sách là hạt đầu tiên tối ưu. Cách này đúng nhưng tốn `O(n log n)` thời gian và `O(n)` bộ nhớ chỉ để lấy **một** giá trị lớn nhất — `max` làm việc đó trong `O(n)`. Ta đưa ra để thấy: khi chỉ cần "phần tử tốt nhất", đừng sắp xếp.

Ngoài ra sắp xếp cho ta cái lợi phụ: biết **thứ tự ưu tiên** của các hạt đầu tiên (hạt tốt thứ hai, thứ ba...), hữu ích nếu đề thay đổi (ví dụ hạt tốt nhất bị cấm).

In [16]:
def min_distance_sorted(height, width, tree, squirrel, nuts):
    tr, tc = tree
    sr, sc = squirrel
    items = []
    for r, c in nuts:
        a = abs(r - tr) + abs(c - tc)
        b = abs(r - sr) + abs(c - sc)
        items.append((a - b, a))          # (tiết kiệm, a)
    items.sort(reverse=True)              # tiết kiệm lớn nhất đứng đầu
    total = 2 * sum(a for _, a in items)
    return total - items[0][0]

for h, w, tree, sq, nuts, expected in examples:
    print(min_distance_sorted(h, w, tree, sq, nuts), expected)

12 12
3 3


**Code nộp LeetCode (Cách 5):**

In [17]:
class SolutionSorted:
    def minDistance(self, height: int, width: int, tree: List[int], squirrel: List[int], nuts: List[List[int]]) -> int:
        tr, tc = tree
        sr, sc = squirrel
        savings = []
        total = 0
        for r, c in nuts:
            a = abs(r - tr) + abs(c - tc)
            b = abs(r - sr) + abs(c - sc)
            savings.append(a - b)
            total += 2 * a
        savings.sort(reverse=True)
        return total - savings[0]

print(SolutionSorted().minDistance(1, 3, [0, 1], [0, 0], [[0, 2]]))   # 3

3


**Độ phức tạp (Cách 5):** Time `O(n log n)`, Space `O(n)`.

## 8. TRACE từng bước

Trace Ví dụ 1 bằng bảng. `tree = (2,2)`, `squirrel = (4,4)`. Ta đi qua từng hạt (`S` cộng dồn là `2·Σa`, `saving` là max của `a − b`):

| hạt | a = d(cây,hạt) | b = d(sóc,hạt) | total sau hạt này (cộng 2a) | a − b | saving sau hạt này |
|---|---|---|---|---|---|
| (3,0) | 3 | 5 | 6 | −2 | −2 |
| (2,5) | 3 | 3 | 12 | 0 | 0 |

Cuối cùng `total − saving = 12 − 0 = 12`. Đúng với đáp án.

Hàm `trace_squirrel` dưới đây in đúng bảng này cho input bất kỳ, kèm kết luận hạt đầu tối ưu.

In [18]:
def trace_squirrel(tree, squirrel, nuts):
    tr, tc = tree
    sr, sc = squirrel
    total = 0
    saving = float("-inf")
    best_idx = -1
    print(f"cây = {tuple(tree)}, sóc = {tuple(squirrel)}")
    print(f"{'hạt':>8} {'a':>3} {'b':>3} {'total':>6} {'a-b':>4} {'saving':>7}")
    for i, (r, c) in enumerate(nuts):
        a = abs(r - tr) + abs(c - tc)
        b = abs(r - sr) + abs(c - sc)
        total += 2 * a
        if a - b > saving:
            saving = a - b
            best_idx = i
        print(f"{str((r, c)):>8} {a:>3} {b:>3} {total:>6} {a - b:>4} {saving:>7}")
    print("hạt đầu tối ưu: chỉ số", best_idx, "->", tuple(nuts[best_idx]))
    print("đáp án = total - saving =", total, "-", saving, "=", total - saving)
    return total - saving

trace_squirrel([2, 2], [4, 4], [[3, 0], [2, 5]])

cây = (2, 2), sóc = (4, 4)
     hạt   a   b  total  a-b  saving
  (3, 0)   3   5      6   -2      -2
  (2, 5)   3   3     12    0       0
hạt đầu tối ưu: chỉ số 1 -> (2, 5)
đáp án = total - saving = 12 - 0 = 12


12

Trace thứ hai: một ví dụ có 3 hạt (đây cũng là phản ví dụ cho cách chọn "hạt gần sóc nhất" ở mục 13), cây `(2,2)`, sóc `(4,4)`, hạt `(1,4), (6,6), (6,0)`.

In [19]:
ans = trace_squirrel([2, 2], [4, 4], [[1, 4], [6, 6], [6, 0]])
print("brute force:", min_distance_brute(7, 7, [2, 2], [4, 4], [[1, 4], [6, 6], [6, 0]]))

cây = (2, 2), sóc = (4, 4)
     hạt   a   b  total  a-b  saving
  (1, 4)   3   3      6    0       0
  (6, 6)   8   4     22    4       4
  (6, 0)   6   6     34    0       4
hạt đầu tối ưu: chỉ số 1 -> (6, 6)
đáp án = total - saving = 34 - 4 = 30
brute force: 30


Ta cũng có thể **mô phỏng đúng từng bước đi** của sóc để thấy tổng số bước hình thành thế nào, với thứ tự: hạt tối ưu trước, rồi các hạt còn lại.

In [20]:
def trace_path(tree, squirrel, nuts, first):
    order = [first] + [i for i in range(len(nuts)) if i != first]
    pos = list(squirrel)
    steps = 0
    for turn, idx in enumerate(order, start=1):
        nut = nuts[idx]
        go = manhattan(pos, nut)
        back = manhattan(nut, tree)
        steps += go + back
        print(f"chuyến {turn}: {tuple(pos)} -> hạt {tuple(nut)} ({go} bước) -> cây ({back} bước) | tổng {steps}")
        pos = list(tree)
    return steps

print(trace_path([2, 2], [4, 4], [[3, 0], [2, 5]], first=1))   # 12
print(trace_path([2, 2], [4, 4], [[3, 0], [2, 5]], first=0))   # 14

chuyến 1: (4, 4) -> hạt (2, 5) (3 bước) -> cây (3 bước) | tổng 6
chuyến 2: (2, 2) -> hạt (3, 0) (3 bước) -> cây (3 bước) | tổng 12
12
chuyến 1: (4, 4) -> hạt (3, 0) (5 bước) -> cây (3 bước) | tổng 8
chuyến 2: (2, 2) -> hạt (2, 5) (3 bước) -> cây (3 bước) | tổng 14
14


## 9. Code nộp LeetCode

Đây là lời giải chuẩn (Cách 3: công thức hai lượt), đúng `class Solution` và signature của bài. Khi nộp lên LeetCode, `List` đã được import sẵn; trong notebook ta import ở đầu.

In [21]:
from typing import List

class Solution:
    def minDistance(self, height: int, width: int, tree: List[int], squirrel: List[int], nuts: List[List[int]]) -> int:
        tr, tc = tree
        sr, sc = squirrel
        # s: nếu MỌI hạt đều tốn khứ hồi cây-hạt-cây
        s = sum(abs(r - tr) + abs(c - tc) for r, c in nuts) * 2
        ans = float("inf")
        for r, c in nuts:
            a = abs(r - tr) + abs(c - tc)      # hạt -> cây
            b = abs(r - sr) + abs(c - sc)      # sóc -> hạt
            ans = min(ans, s - a + b)          # hạt này được nhặt đầu tiên
        return ans

sol = Solution()
for h, w, tree, sq, nuts, expected in examples:
    got = sol.minDistance(h, w, tree, sq, nuts)
    print(got, expected, "OK" if got == expected else "SAI")

12 12 OK
3 3 OK


## 10. Vì sao đúng

**Bước 1 — Mô hình hoá.** Một lịch trình nhặt hạt là một hoán vị `π` của các hạt. Chi phí:

```
cost(π) = d(S, nut_π1) + d(nut_π1, T) + Σ_{j≥2} [ d(T, nut_πj) + d(nut_πj, T) ]
```

Vì đường đi ngắn nhất giữa hai ô là khoảng cách Manhattan (không có vật cản) và khoảng cách đối xứng, `d(T, x) = d(x, T) = a_x`.

**Bước 2 — Chi phí chỉ phụ thuộc hạt đầu.** Viết lại:

```
cost(π) = b_{π1} + a_{π1} + 2·Σ_{j≥2} a_{πj}
        = b_{π1} + a_{π1} + (S − 2·a_{π1})
        = S − a_{π1} + b_{π1}
```

(`S = 2·Σ_i a_i`). Thứ tự của các hạt còn lại **biến mất** khỏi công thức.

**Bước 3 — Tối ưu.** `min_π cost(π) = min_k (S − a_k + b_k)` vì mọi hạt `k` đều có thể đứng đầu một hoán vị nào đó. Đó chính xác là điều code tính.

**Bước 4 — Tại sao không thể làm tốt hơn.** Mỗi hạt phải được đem về cây ít nhất một lần, nên mỗi hạt tốn ít nhất quãng `hạt → cây` ở đoạn cuối. Quãng đi tới hạt đó (`cây → hạt` hoặc `sóc → hạt`) cũng bắt buộc. Không có hạt nào được "nhặt dọc đường" vì sóc chỉ mang tối đa một hạt. Vì thế tổng ở Bước 2 đã là tối thiểu cho mỗi hạt đầu cụ thể.

Cell dưới kiểm chứng đẳng thức ở Bước 2 trên nhiều input ngẫu nhiên nhỏ: **chi phí của từng hoán vị** đúng bằng `S − a_{π1} + b_{π1}`.

In [22]:
random.seed(573)
checked = 0
for _ in range(300):
    n = random.randint(1, 5)
    tree = [random.randint(0, 6), random.randint(0, 6)]
    sq = [random.randint(0, 6), random.randint(0, 6)]
    nuts = [[random.randint(0, 6), random.randint(0, 6)] for _ in range(n)]
    S_ = 2 * sum(manhattan(tree, x) for x in nuts)
    for order in permutations(range(n)):
        cost = 0
        pos = sq
        for idx in order:
            cost += manhattan(pos, nuts[idx]) + manhattan(nuts[idx], tree)
            pos = tree
        k = order[0]
        assert cost == S_ - manhattan(tree, nuts[k]) + manhattan(sq, nuts[k])
        checked += 1
print("đẳng thức cost(π) = S - a + b đúng cho", checked, "hoán vị")

đẳng thức cost(π) = S - a + b đúng cho 8332 hoán vị


## 11. Time / Space Complexity

Với `n = len(nuts)`:

- **Time:** `O(n)` — một lượt `sum` và một vòng `for`, mỗi hạt xử lý `O(1)` (vài phép trừ, `abs`, cộng). Không phụ thuộc `height`, `width`.
- **Space:** `O(1)` — chỉ vài biến số nguyên. Generator trong `sum(... for ...)` không tạo danh sách.

Về độ lớn số: mỗi khoảng cách tối đa khoảng `height + width ≤ 200`, nhân 5000 hạt, nhân 2 → khoảng `2 × 10^6`, vừa trong `int` 32-bit của các ngôn ngữ khác (Python thì không lo tràn số).

In [23]:
# Đo nhanh: n = 5000 hạt (mức tối đa của đề) chạy tức thì với công thức, còn O(n^2) thì chậm hơn nhiều
import time
random.seed(5000)
n = 5000
tree = [random.randint(0, 100), random.randint(0, 100)]
sq = [random.randint(0, 100), random.randint(0, 100)]
nuts = [[random.randint(0, 100), random.randint(0, 100)] for _ in range(n)]

t0 = time.perf_counter(); r1 = SolutionFormula().minDistance(100, 100, tree, sq, nuts); t1 = time.perf_counter()
r2 = SolutionTryFirst().minDistance(100, 100, tree, sq, nuts); t2 = time.perf_counter()
assert r1 == r2
print("hai cách cùng đáp án:", r1)
print("công thức O(n) nhanh hơn thử-từng-hạt O(n^2):", (t1 - t0) < (t2 - t1))

hai cách cùng đáp án: 547443
công thức O(n) nhanh hơn thử-từng-hạt O(n^2): True


## 12. Bộ test

Gồm: ví dụ của đề, edge case, phản ví dụ cho cách tham lam sai, đối chiếu các cách nhiều input bằng `assert`, và test ngẫu nhiên có seed đúng ràng buộc.

In [24]:
sol = Solution()

# 1) Ví dụ của đề
assert sol.minDistance(5, 7, [2, 2], [4, 4], [[3, 0], [2, 5]]) == 12
assert sol.minDistance(1, 3, [0, 1], [0, 0], [[0, 2]]) == 3
print("ví dụ đề OK")

# 2) Edge case
# 2a) một hạt: tổng = d(sóc,hạt) + d(hạt,cây)
assert sol.minDistance(10, 10, [5, 5], [0, 0], [[9, 9]]) == 18 + 8
# 2b) hạt nằm đúng ở cây (a = 0): chỉ cần sóc đi tới cây
assert sol.minDistance(10, 10, [5, 5], [0, 0], [[5, 5]]) == 10
# 2c) sóc đứng ngay ở cây
assert sol.minDistance(10, 10, [3, 3], [3, 3], [[3, 4], [0, 3]]) == 2 * (1 + 3)
# 2d) sóc đứng ngay trên hạt: b = 0
assert sol.minDistance(10, 10, [0, 0], [4, 4], [[4, 4]]) == 8
# 2e) nhiều hạt trùng vị trí
assert sol.minDistance(10, 10, [0, 0], [1, 0], [[1, 0], [1, 0], [1, 0]]) == 0 + 1 + 2 * 1 * 2
# 2f) lưới 1x1, mọi thứ ở (0,0)
assert sol.minDistance(1, 1, [0, 0], [0, 0], [[0, 0]]) == 0
# 2g) toạ độ ở cạnh (<= height, width)
assert sol.minDistance(100, 100, [0, 0], [100, 100], [[100, 100]]) == 200
print("edge case OK")

ví dụ đề OK
edge case OK


In [25]:
# 3) Phản ví dụ cho cách tham lam: 'hạt gần sóc nhất đi trước' (xem mục 13)
tree, sq = [2, 2], [4, 4]
nuts = [[1, 4], [6, 6], [6, 0]]
assert sol.minDistance(7, 7, tree, sq, nuts) == 30
assert min_distance_brute(7, 7, tree, sq, nuts) == 30
print("phản ví dụ: đáp án đúng 30 (xem mục 13 để biết cách tham lam cho 34)")

# 4) Đối chiếu 5 cách + brute force trên nhiều input nhỏ
random.seed(2024)
impls = [min_distance_try_first, min_distance_formula, min_distance_one_pass, min_distance_sorted]
classes = [SolutionTryFirst(), SolutionFormula(), SolutionOnePass(), SolutionSorted(), Solution()]
for _ in range(1500):
    n = random.randint(1, 6)
    H, W = random.randint(1, 8), random.randint(1, 8)
    tree = [random.randint(0, H), random.randint(0, W)]
    sq = [random.randint(0, H), random.randint(0, W)]
    nuts = [[random.randint(0, H), random.randint(0, W)] for _ in range(n)]
    expected = min_distance_brute(H, W, tree, sq, nuts)
    for f in impls:
        assert f(H, W, tree, sq, nuts) == expected, (f.__name__, tree, sq, nuts)
    for s in classes:
        assert s.minDistance(H, W, tree, sq, nuts) == expected
print("1500 input nhỏ: mọi cách khớp brute force")

phản ví dụ: đáp án đúng 30 (xem mục 13 để biết cách tham lam cho 34)


1500 input nhỏ: mọi cách khớp brute force


In [26]:
# 5) Ngẫu nhiên lớn có seed, đúng ràng buộc (n tới 5000, toạ độ 0..100); so các cách O(n) / O(n log n) / O(n^2)
random.seed(99)
for trial in range(6):
    n = random.choice([1, 2, 50, 300, 1000])
    H, W = random.randint(1, 100), random.randint(1, 100)
    tree = [random.randint(0, H), random.randint(0, W)]
    sq = [random.randint(0, H), random.randint(0, W)]
    nuts = [[random.randint(0, H), random.randint(0, W)] for _ in range(n)]
    results = {s.__class__.__name__: s.minDistance(H, W, tree, sq, nuts) for s in classes}
    assert len(set(results.values())) == 1, results
    print(f"trial {trial}: n={n:>4}, đáp án = {sol.minDistance(H, W, tree, sq, nuts)}")

# 6) Test tính chất: đáp án >= tổng khứ hồi tối thiểu và <= S + khoảng cách xa nhất
random.seed(7)
for _ in range(200):
    n = random.randint(1, 30)
    tree = [random.randint(0, 20), random.randint(0, 20)]
    sq = [random.randint(0, 20), random.randint(0, 20)]
    nuts = [[random.randint(0, 20), random.randint(0, 20)] for _ in range(n)]
    ans = sol.minDistance(20, 20, tree, sq, nuts)
    S_ = 2 * sum(manhattan(tree, x) for x in nuts)
    # sóc cách cây d(S,T): đi từ sóc tới hạt rồi về cây không quá d(S,T) lệch so với từ cây
    assert ans >= S_ - max(manhattan(tree, x) for x in nuts)
    assert ans <= S_ + manhattan(sq, tree)       # tam giác: b <= d(S,T) + a
print("test tính chất OK")

trial 0: n= 300, đáp án = 15006
trial 1: n=   1, đáp án = 92
trial 2: n= 300, đáp án = 21455
trial 3: n=  50, đáp án = 3010
trial 4: n= 300, đáp án = 26433
trial 5: n= 300, đáp án = 9069
test tính chất OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên nhân đôi (`2×`) cho các hạt còn lại

Hạt không phải đầu tiên cần **đi từ cây tới hạt rồi quay về cây**, nên tốn `2a`. Nhiều bạn chỉ cộng `a`.

In [27]:
def wrong_no_double(height, width, tree, squirrel, nuts):
    tr, tc = tree
    sr, sc = squirrel
    s = sum(abs(r - tr) + abs(c - tc) for r, c in nuts)          # SAI: thiếu * 2
    return min(s - (abs(r - tr) + abs(c - tc)) + (abs(r - sr) + abs(c - sc)) for r, c in nuts)

print("sai :", wrong_no_double(5, 7, [2, 2], [4, 4], [[3, 0], [2, 5]]), "(kỳ vọng 12)")
print("đúng:", Solution().minDistance(5, 7, [2, 2], [4, 4], [[3, 0], [2, 5]]))

sai : 6 (kỳ vọng 12)
đúng: 12


**Vì sao sai:** công thức đúng là `2·Σa − a_k + b_k`. Bỏ `2×` thì `Σa − a_k + b_k` đếm mỗi hạt chỉ một chiều. **Cách nhớ:** mỗi hạt (trừ hạt đầu) là *khứ hồi*.

### Lỗi 2 — Tham lam: chọn hạt gần sóc nhất làm hạt đầu

Nghe rất tự nhiên: "sóc ở đâu thì nhặt hạt gần mình nhất trước". Nhưng đoạn `hạt → cây` mới là quan trọng: hạt đầu **tiết kiệm** `a − b`, không phải chỉ phụ thuộc `b`.

In [28]:
def wrong_nearest_to_squirrel(height, width, tree, squirrel, nuts):
    nearest = min(nuts, key=lambda p: manhattan(p, squirrel))     # SAI: chỉ nhìn b
    S_ = 2 * sum(manhattan(p, tree) for p in nuts)
    return S_ - manhattan(nearest, tree) + manhattan(nearest, squirrel)

tree, sq = [2, 2], [4, 4]
nuts = [[1, 4], [6, 6], [6, 0]]
print("tham lam gần sóc nhất:", wrong_nearest_to_squirrel(7, 7, tree, sq, nuts))   # 34
print("đúng                :", Solution().minDistance(7, 7, tree, sq, nuts))        # 30

tham lam gần sóc nhất: 34
đúng                : 30


**Vì sao sai:** hạt `(1,4)` gần sóc nhất (`b = 3`) nhưng cũng gần cây (`a = 3`): tiết kiệm `a − b = 0`. Hạt `(6,6)` có `b = 4`, `a = 8`: tiết kiệm `4`, tốt hơn. Phải so **`a − b`**, không phải `b`.

Tương tự, "chọn hạt gần cây nhất đi trước" cũng sai vì quên `b`.

### Lỗi 3 — Chỉ so `b` mà không trừ `a` (hoặc không cộng `S`)

Một cách làm sai khác: tính `min(b_k) + S`. Như vậy hạt đầu được tính `S + b` thay vì `S − a + b`: thừa `a`.

In [29]:
def wrong_min_b_only(height, width, tree, squirrel, nuts):
    tr, tc = tree
    sr, sc = squirrel
    s = sum(abs(r - tr) + abs(c - tc) for r, c in nuts) * 2
    return s + min(abs(r - sr) + abs(c - sc) for r, c in nuts)       # SAI: quên - a

print("sai :", wrong_min_b_only(1, 3, [0, 1], [0, 0], [[0, 2]]), "(kỳ vọng 3)")
print("đúng:", Solution().minDistance(1, 3, [0, 1], [0, 0], [[0, 2]]))

sai : 4 (kỳ vọng 3)
đúng: 3


**Vì sao sai:** `S` đã tính hạt đó ở dạng `2a` (khứ hồi), nhưng hạt đầu thực tế tốn `b + a`: phải **gỡ** `2a` ra và **thêm** `b + a`, tức `S − 2a + b + a = S − a + b`.

### Lỗi 4 — Khởi tạo `ans = 0` hoặc `saving = 0` cho bài toán lấy `min`/`max`

Khởi tạo `ans = 0` cho phép tìm `min` làm kết quả luôn là `0`. Khởi tạo `saving = 0` ở Cách 4 sai khi mọi `a − b` đều âm (xem ví dụ cuối mục 7.3).

In [30]:
def wrong_init_zero(height, width, tree, squirrel, nuts):
    tr, tc = tree
    sr, sc = squirrel
    total = 0
    saving = 0                                   # SAI: nên là float("-inf")
    for r, c in nuts:
        a = abs(r - tr) + abs(c - tc)
        b = abs(r - sr) + abs(c - sc)
        total += 2 * a
        saving = max(saving, a - b)
    return total - saving

# hạt sát cây, sóc ở xa: a - b = 1 - 17 = -16 < 0
print("sai :", wrong_init_zero(10, 10, [0, 0], [9, 9], [[0, 1]]), "(kỳ vọng 18)")
print("đúng:", Solution().minDistance(10, 10, [0, 0], [9, 9], [[0, 1]]))

sai : 2 (kỳ vọng 18)
đúng: 18


**Vì sao sai:** `saving = 0` ngầm giả định "có thể không nhặt hạt đầu nào tiết kiệm", nhưng sóc **bắt buộc** nhặt một hạt đầu. Nếu mọi `a − b` đều âm, đáp án thật là `total − (số âm lớn nhất)`, tức lớn hơn `total`.

### Lỗi 5 — Dùng `height`/`width` để dựng lưới và chạy BFS cho từng cặp

Nhiều bạn nghĩ "lưới thì phải BFS". Với lưới **không có vật cản**, đường đi ngắn nhất = Manhattan, BFS chỉ tốn công (và với `n = 5000` hạt thì BFS từng hạt sẽ rất chậm). Đề cố ý cho `height`, `width` là *thông tin nhiễu*. Ở mục 15.2 ta xem khi có tường thì cần BFS thật.

## 14. Mẹo & nhận dạng pattern

- **"Mỗi lần đi một việc rồi quay về điểm xuất phát cố định"** (cây, kho, nhà ga...) → chi phí mọi việc **trừ việc đầu tiên** là khứ hồi. Chỉ cần xét "việc đầu tiên đặc biệt thế nào".
- **Đổi biến thể sang "tiết kiệm":** thay vì tính tổng từng phương án, tính tổng lý tưởng rồi trừ phần tiết kiệm lớn nhất. Dạng `đáp án = S − max(gain)` rất hay gặp.
- **Kiểm chứng bằng vét cạn nhỏ:** bài trông giống TSP (`n!` thứ tự) nhưng công thức cho ra đáp án; luôn viết brute force cho `n ≤ 6` để thử lại.
- **Không dùng thông tin thừa:** `height`, `width` không xuất hiện trong lời giải. Học cách bỏ qua dữ liệu không cần.
- **Manhattan thay BFS** khi lưới không có vật cản và cho phép đi 4 hướng.
- Đừng tham lam theo một đại lượng đơn lẻ (`b`); hãy xét **độ chênh** (`a − b`).

## 15. Biến thể

### 15.1. Trả về hạt đầu tiên tối ưu (chỉ số), không chỉ khoảng cách

Chỉ cần nhớ chỉ số tại thời điểm cập nhật `saving` (giống hàm trace ở mục 8). Nếu có nhiều hạt hoà, ta trả về chỉ số nhỏ nhất.

In [31]:
def best_first_nut(tree, squirrel, nuts):
    tr, tc = tree
    sr, sc = squirrel
    best_gain = float("-inf")
    best_idx = -1
    for i, (r, c) in enumerate(nuts):
        gain = (abs(r - tr) + abs(c - tc)) - (abs(r - sr) + abs(c - sc))
        if gain > best_gain:                 # '>' giữ chỉ số nhỏ nhất khi hoà
            best_gain, best_idx = gain, i
    return best_idx

print(best_first_nut([2, 2], [4, 4], [[3, 0], [2, 5]]))               # 1
print(best_first_nut([2, 2], [4, 4], [[1, 4], [6, 6], [6, 0]]))        # 1

1
1


### 15.2. Lưới có tường: bắt buộc dùng BFS

Nếu có ô `#` là tường, khoảng cách giữa hai ô không còn là Manhattan. Công thức `S − a_k + b_k` **vẫn đúng** miễn là `a_k = dist(cây, hạt k)` và `b_k = dist(sóc, hạt k)` là khoảng cách thật (đối xứng, đường ngắn nhất). Ta chỉ cần **hai lần BFS**: một từ cây, một từ sóc, mỗi lần BFS tính khoảng cách tới mọi ô. Hạt không tới được cây hoặc sóc không tới được hạt đầu → bỏ qua/`-1`.

In [32]:
from collections import deque

def bfs_dist(grid, start):
    # grid: list[str], '#' là tường; trả về dict (r,c) -> khoảng cách
    H, W = len(grid), len(grid[0])
    dist = {tuple(start): 0}
    q = deque([tuple(start)])
    while q:
        r, c = q.popleft()
        for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            nr, nc = r + dr, c + dc
            if 0 <= nr < H and 0 <= nc < W and grid[nr][nc] != "#" and (nr, nc) not in dist:
                dist[(nr, nc)] = dist[(r, c)] + 1
                q.append((nr, nc))
    return dist

def min_distance_walls(grid, tree, squirrel, nuts):
    dt = bfs_dist(grid, tree)
    ds = bfs_dist(grid, squirrel)
    if any(tuple(n) not in dt for n in nuts):
        return -1                                       # có hạt không thể đem về cây
    S_ = 2 * sum(dt[tuple(n)] for n in nuts)
    best = float("inf")
    for n in nuts:
        n = tuple(n)
        if n in ds:                                     # sóc phải tới được hạt đầu
            best = min(best, S_ - dt[n] + ds[n])
    return best if best != float("inf") else -1

grid = [
    "....#..",
    ".##.#..",
    "....#..",
    ".##...."
]
# không có tường nên BFS = Manhattan (kiểm chứng)
open_grid = ["." * 7 for _ in range(5)]
print(min_distance_walls(open_grid, [2, 2], [4, 4], [[3, 0], [2, 5]]))   # 12 (khớp Manhattan)

# có tường thì đáp án lớn hơn hoặc bằng Manhattan
print(min_distance_walls(grid, [0, 0], [3, 6], [[0, 6], [2, 5]]))
print(Solution().minDistance(4, 7, [0, 0], [3, 6], [[0, 6], [2, 5]]))

12
33
21


Kiểm chứng biến thể này với brute-force hoán vị trên lưới nhỏ ngẫu nhiên có tường (dùng khoảng cách BFS làm `d`).

In [33]:
def brute_walls(grid, tree, squirrel, nuts):
    dt = bfs_dist(grid, tree)
    ds = bfs_dist(grid, squirrel)
    # khoảng cách giữa mọi cặp ô cần thiết: ta chỉ cần cây<->hạt và sóc->hạt (đối xứng)
    INF = float("inf")
    best = INF
    for order in permutations(range(len(nuts))):
        cost = 0
        ok = True
        for j, idx in enumerate(order):
            n = tuple(nuts[idx])
            if n not in dt:
                ok = False
                break
            cost += dt[n] * 2 if j > 0 else (ds.get(n, INF) + dt[n])
        if ok:
            best = min(best, cost)
    return -1 if best == INF else best

random.seed(15)
for _ in range(300):
    H, W = random.randint(2, 5), random.randint(2, 5)
    g = ["".join("#" if random.random() < 0.25 else "." for _ in range(W)) for _ in range(H)]
    cells = [(r, c) for r in range(H) for c in range(W) if g[r][c] == "."]
    if len(cells) < 3:
        continue
    tree = list(random.choice(cells))
    sq = list(random.choice(cells))
    nuts = [list(random.choice(cells)) for _ in range(random.randint(1, 4))]
    assert min_distance_walls(g, tree, sq, nuts) == brute_walls(g, tree, sq, nuts), (g, tree, sq, nuts)
print("biến thể có tường: công thức + BFS khớp brute force")

biến thể có tường: công thức + BFS khớp brute force


### 15.3. Nhiều truy vấn: nhiều vị trí xuất phát của sóc

Nếu cho `q` vị trí sóc khác nhau (cùng cây và hạt), ta **tính `S` và các `a_i` một lần**, rồi mỗi truy vấn chỉ tốn `O(n)` để tìm `max(a_i − b_i)`. Tổng `O(n + q·n)` thay vì tính lại `S` mỗi lần. (Có thể làm nhanh hơn nữa bằng cách tách `|r − sr| + |c − sc|` thành 4 trường hợp dấu và giữ 4 giá trị tối ưu — một kỹ thuật quen thuộc với khoảng cách Manhattan, để dành cho bạn tự tìm hiểu.)

### 15.4. Sóc mang được hai hạt mỗi lần (rất khó)

Khi sức chứa là 2, bài trở thành bài ghép cặp hạt tối ưu (tương tự bài ghép cặp, có dạng quy hoạch động bitmask hoặc luồng/ghép cặp trọng số). Công thức đơn giản ở đây **không còn đúng**; chỉ nhắc để thấy giả thiết "tối đa một hạt" quan trọng thế nào.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn hoán vị | Thử mọi thứ tự nhặt, mô phỏng | `O(n! · n)` | `O(n)` | Chỉ làm oracle cho `n ≤ 7` để kiểm tra |
| 2. Thử từng hạt đầu | `n` lựa chọn hạt đầu, cộng lại `2a` các hạt khác | `O(n²)` | `O(1)` | Khi mới nhận ra "chỉ hạt đầu quan trọng", chưa rút gọn |
| 3. Công thức hai lượt | `S = 2Σa`, trả `min(S − a + b)` | `O(n)` | `O(1)` | **Lời giải chuẩn nộp LeetCode** |
| 4. Một lượt, giữ saving | `Σ2a − max(a − b)` trong một vòng | `O(n)` | `O(1)` | Dữ liệu dạng luồng, đọc một lần |
| 5. Sắp xếp theo saving | Sort `a − b` giảm dần, lấy đầu | `O(n log n)` | `O(n)` | Khi cần thứ tự ưu tiên các hạt đầu; còn không thì thừa |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output của ba dòng sau, rồi chạy kiểm tra:

```python
print(manhattan([1, 1], [4, 5]))
print(sum(abs(r - 0) + abs(c - 0) for r, c in [[1, 2], [3, 0], [0, 4]]) * 2)
print(Solution().minDistance(3, 3, [0, 0], [0, 0], [[1, 1]]))
```

In [34]:
# Bài 1: viết dự đoán của bạn vào các biến bên dưới (chưa chạy code mẫu)
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Cho `tree = [1, 1]`, `squirrel = [0, 0]`, `nuts = [[0, 1], [2, 2]]`. Hãy tính tay `a`, `b` của từng hạt, `S`, rồi đáp án. Điền vào các biến.

In [35]:
# Bài 2: điền kết quả tính tay
hand_S = None        # S = 2 * tổng a
hand_answer = None   # đáp án

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành hàm theo công thức hai lượt.

In [36]:
def exercise_3_min_distance(tree: List[int], squirrel: List[int], nuts: List[List[int]]) -> int:
    tr, tc = tree
    sr, sc = squirrel
    # TODO 1: s = 2 * (tổng khoảng cách Manhattan từ mỗi hạt tới cây)
    # TODO 2: ans = vô cực dương
    # TODO 3: với mỗi hạt tính a (tới cây) và b (từ sóc), cập nhật ans = min(ans, s - a + b)
    # TODO 4: return ans
    return None


# assert exercise_3_min_distance([2, 2], [4, 4], [[3, 0], [2, 5]]) == 12
# assert exercise_3_min_distance([0, 1], [0, 0], [[0, 2]]) == 3
# assert exercise_3_min_distance([5, 5], [0, 0], [[5, 5]]) == 10

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 4, hãy tự viết `exercise_4_one_pass(tree, squirrel, nuts)` chỉ dùng **một** vòng `for`, giữ `total` và `saving`. Nhớ khởi tạo `saving` đúng.

In [37]:
def exercise_4_one_pass(tree: List[int], squirrel: List[int], nuts: List[List[int]]) -> int:
    # TODO
    pass


# assert exercise_4_one_pass([2, 2], [4, 4], [[3, 0], [2, 5]]) == 12
# assert exercise_4_one_pass([0, 0], [9, 9], [[0, 1]]) == 18
# assert exercise_4_one_pass([2, 2], [4, 4], [[1, 4], [6, 6], [6, 0]]) == 30

### Bài 5 — Mini challenge: nhiều vị trí xuất phát của sóc (khó)

Viết `exercise_5_queries(tree, nuts, squirrels)` trả về danh sách đáp án, mỗi phần tử ứng với một vị trí sóc trong `squirrels`. Yêu cầu: tính `S` và danh sách `a` **một lần** rồi dùng lại; mỗi truy vấn `O(n)`.

```python
exercise_5_queries([2, 2], [[3, 0], [2, 5]], [[4, 4], [2, 2], [0, 0]])   # trả về một list 3 số
```

Hãy tự tính tay đáp án cho từng vị trí sóc trước (ví dụ sóc ở `[4, 4]` là Ví dụ 1 nên đáp án `12`), rồi so với brute force ở mục 5 khi kiểm tra ở mục 19.

In [38]:
def exercise_5_queries(tree: List[int], nuts: List[List[int]], squirrels: List[List[int]]) -> List[int]:
    # TODO
    pass


# kết quả đúng được kiểm tra ở mục 19 bằng brute force

## 18. Gợi ý 3 mức

**Bài 3 / Bài 4**

- *Mức 1 (nhẹ):* Sau chuyến đầu tiên, sóc luôn ở đâu? Từ đó mỗi hạt còn lại tốn bao nhiêu?
- *Mức 2 (vừa):* Nếu mọi hạt đều tốn `2a`, tổng là `S`. Chọn hạt `k` làm hạt đầu thì thay `2a_k` bằng cái gì?
- *Mức 3 (gần đáp án):* `tổng(k) = S − a_k + b_k`. Với một lượt: giữ `total = Σ 2a` và `saving = max(a − b)` với `saving` khởi tạo `-inf`; trả `total − saving`.

**Bài 5**

- *Mức 1:* Phần nào của lời giải không phụ thuộc vị trí sóc?
- *Mức 2:* `S` và `a_i` không phụ thuộc sóc. Chỉ `b_i` thay đổi theo từng truy vấn.
- *Mức 3:* Với mỗi sóc `(sr, sc)`: `S + min(|r − sr| + |c − sc| − a_i)` qua các hạt. Gọi lại công thức một lần cho từng sóc, dùng chung `S`, `a`.

## 19. Đáp án tham khảo

### Đáp án Bài 1

- Dòng 1: `|1-4| + |1-5| = 3 + 4 = 7`.
- Dòng 2: `(1+2) + (3+0) + (0+4) = 10`, nhân 2 = `20`.
- Dòng 3: một hạt tại `(1,1)`, sóc và cây cùng ở `(0,0)`: `b = 2`, `a = 2` → `2 + 2 = 4`.

In [39]:
assert manhattan([1, 1], [4, 5]) == 7
assert sum(abs(r - 0) + abs(c - 0) for r, c in [[1, 2], [3, 0], [0, 4]]) * 2 == 20
assert Solution().minDistance(3, 3, [0, 0], [0, 0], [[1, 1]]) == 4
print("Bài 1 OK")

Bài 1 OK


### Đáp án Bài 2

`tree = (1,1)`, sóc `(0,0)`. Hạt `(0,1)`: `a = 1`, `b = 1`. Hạt `(2,2)`: `a = 2`, `b = 4`. `S = 2·(1 + 2) = 6`. Hạt đầu `(0,1)`: `6 − 1 + 1 = 6`; hạt đầu `(2,2)`: `6 − 2 + 4 = 8`. Đáp án `6`.

In [40]:
hand_S, hand_answer = 6, 6
assert hand_S == 2 * sum(manhattan([1, 1], n) for n in [[0, 1], [2, 2]])
assert hand_answer == Solution().minDistance(3, 3, [1, 1], [0, 0], [[0, 1], [2, 2]])
assert hand_answer == min_distance_brute(3, 3, [1, 1], [0, 0], [[0, 1], [2, 2]])
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [41]:
def exercise_3_solution(tree: List[int], squirrel: List[int], nuts: List[List[int]]) -> int:
    tr, tc = tree
    sr, sc = squirrel
    s = 2 * sum(abs(r - tr) + abs(c - tc) for r, c in nuts)
    ans = float("inf")
    for r, c in nuts:
        a = abs(r - tr) + abs(c - tc)
        b = abs(r - sr) + abs(c - sc)
        ans = min(ans, s - a + b)
    return ans

assert exercise_3_solution([2, 2], [4, 4], [[3, 0], [2, 5]]) == 12
assert exercise_3_solution([0, 1], [0, 0], [[0, 2]]) == 3
assert exercise_3_solution([5, 5], [0, 0], [[5, 5]]) == 10
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [42]:
def exercise_4_solution(tree: List[int], squirrel: List[int], nuts: List[List[int]]) -> int:
    tr, tc = tree
    sr, sc = squirrel
    total = 0
    saving = float("-inf")
    for r, c in nuts:
        a = abs(r - tr) + abs(c - tc)
        b = abs(r - sr) + abs(c - sc)
        total += 2 * a
        saving = max(saving, a - b)
    return total - saving

assert exercise_4_solution([2, 2], [4, 4], [[3, 0], [2, 5]]) == 12
assert exercise_4_solution([0, 0], [9, 9], [[0, 1]]) == 18
assert exercise_4_solution([2, 2], [4, 4], [[1, 4], [6, 6], [6, 0]]) == 30
random.seed(4)
for _ in range(300):
    n = random.randint(1, 5)
    tr_ = [random.randint(0, 6), random.randint(0, 6)]
    sq_ = [random.randint(0, 6), random.randint(0, 6)]
    ns_ = [[random.randint(0, 6), random.randint(0, 6)] for _ in range(n)]
    assert exercise_4_solution(tr_, sq_, ns_) == min_distance_brute(6, 6, tr_, sq_, ns_)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

Với sóc `[4,4]` đáp án là `12` (Ví dụ 1); sóc đứng ngay ở cây `[2,2]` thì đáp án bằng `S = 12`; sóc ở `[0,0]`: hạt `(3,0)` có `b = 3`, hạt `(2,5)` có `b = 7`, nên `S − a + b` lần lượt là `12 − 3 + 3 = 12` và `12 − 3 + 7 = 16` → `12`. Cell dưới tính cả ba và so với brute force.

In [43]:
def exercise_5_solution(tree: List[int], nuts: List[List[int]], squirrels: List[List[int]]) -> List[int]:
    tr, tc = tree
    a_list = [abs(r - tr) + abs(c - tc) for r, c in nuts]      # tính một lần
    S_ = 2 * sum(a_list)                                       # tính một lần
    answers = []
    for sr, sc in squirrels:
        best = float("inf")
        for (r, c), a in zip(nuts, a_list):
            best = min(best, S_ - a + abs(r - sr) + abs(c - sc))
        answers.append(best)
    return answers

res = exercise_5_solution([2, 2], [[3, 0], [2, 5]], [[4, 4], [2, 2], [0, 0]])
print(res)
expected = [min_distance_brute(5, 7, [2, 2], s, [[3, 0], [2, 5]]) for s in ([4, 4], [2, 2], [0, 0])]
assert res == expected
assert res == [12, 12, 12]

random.seed(55)
for _ in range(200):
    n = random.randint(1, 5)
    tr_ = [random.randint(0, 6), random.randint(0, 6)]
    ns_ = [[random.randint(0, 6), random.randint(0, 6)] for _ in range(n)]
    qs = [[random.randint(0, 6), random.randint(0, 6)] for _ in range(4)]
    assert exercise_5_solution(tr_, ns_, qs) == [Solution().minDistance(6, 6, tr_, q, ns_) for q in qs]
print("Bài 5 OK")

[12, 12, 12]
Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template

```python
class Solution:
    def minDistance(self, height: int, width: int, tree: List[int], squirrel: List[int], nuts: List[List[int]]) -> int:
        tr, tc = tree
        sr, sc = squirrel
        s = sum(abs(r - tr) + abs(c - tc) for r, c in nuts) * 2     # mọi hạt khứ hồi
        ans = float("inf")
        for r, c in nuts:
            a = abs(r - tr) + abs(c - tc)       # hạt -> cây
            b = abs(r - sr) + abs(c - sc)       # sóc -> hạt
            ans = min(ans, s - a + b)           # hạt này nhặt đầu tiên
        return ans
```

Câu thần chú: **"Tổng khứ hồi, trừ a, cộng b, chọn nhỏ nhất."**

### Checklist trước khi submit

- [ ] Có nhân đôi `2×` cho tổng khoảng cách hạt → cây.
- [ ] Công thức là `s - a + b` (không phải `s + b`, không phải `s - b`).
- [ ] Khoảng cách dùng Manhattan `abs(dr) + abs(dc)`; không dựng lưới/BFS.
- [ ] `ans` khởi tạo `float("inf")` (hoặc `saving` khởi tạo `-inf`), không phải `0`.
- [ ] Đã thử `n = 1`, hạt trùng cây, sóc trùng cây, sóc trùng hạt, hạt trùng nhau.
- [ ] Đã đối chiếu brute force hoán vị trên input nhỏ.
- [ ] Skeleton đúng: `class Solution`, method `minDistance(self, height, width, tree, squirrel, nuts)`.
- [ ] Bài Premium: nếu có thể, đối chiếu lại đề gốc (đặc biệt ràng buộc toạ độ).